In [1]:
# Jalankan perintah instalasi ini di Sel 1 Anda
!pip install -q -U transformers accelerate bitsandbytes peft trl datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 108.8 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 48.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 55.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 85.1 MB/s eta 0:00:00:00:01


In [2]:
import torch

# 1. Konfigurasi Model & Data Path
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
DATA_PATH = "/kaggle/input/datasets/zadaniammusthofa/dataset-qlora/dataset_sft_final_ready.jsonl"
OUTPUT_DIR = "/kaggle/working/qlora_qwen_empathy"

# 2. Hyperparameters untuk 100 Data Sintetis (Fokus Style Alignment)
EPOCHS = 3                    # Cukup 3 kali putaran agar tidak overfitting
LEARNING_RATE = 2e-5          # LR rendah untuk menjaga pengetahuan dasar model
MAX_SEQ_LENGTH = 512          # 512 sangat aman untuk dataset kita

# 3. Kalkulasi Batch Size untuk Dual GPU T4
BATCH_SIZE_PER_DEVICE = 2     # 2 baris data per microstep
GRADIENT_ACC_STEPS = 4        # Total efektif batch size = 2 x 4 = 16 data per step

print("Konfigurasi global berhasil disimpan. Siap melangkah ke pemrosesan data.")


Konfigurasi global berhasil disimpan. Siap melangkah ke pemrosesan data.


In [6]:
from datasets import load_dataset
from transformers import AutoTokenizer

# 1. Muat dataset JSONL dari Kaggle Input
dataset = load_dataset("json", data_files=DATA_PATH, split="train")

# 2. Muat Tokenizer Qwen untuk memproses teks
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"  # Wajib 'right' untuk proses autoregressive SFT training

# 3. Fungsi untuk membungkus format chat multi-turn
def format_chat_template(batch):
    formatted_texts = []
    # Loop setiap baris data di dalam batch
    for messages in batch["messages"]:
        # Membawa struktur role (user/assistant) ke format token internal Qwen (<|im_start|>, dll.)
        text = tokenizer.apply_chat_template(messages, tokenize=False)
        formatted_texts.append(text)
    return {"text": formatted_texts}

# 4. Transformasikan dataset
processed_dataset = dataset.map(
    format_chat_template,
    batched=True,
    remove_columns=dataset.column_names # Hapus kolom lama agar menyisakan kolom "text" saja
)

# 5. Intip hasil akhir (Sanity Check)
print("--- CONTOH HASIL FORMAT DATASET ---")
print(processed_dataset[0]["text"])


Generating train split: 0 examples [00:00, ? examples/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

--- CONTOH HASIL FORMAT DATASET ---
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Aku lagi sering banget insecure sama bentuk fisikku akhir-akhir ini. Pas lihat foto-foto di Instagram, temen-temenku kok pada kelihatan glowing dan proporsional semua ya.<|im_end|>
<|im_start|>assistant
Capek banget pasti ya kalau mata kita terus-terusan disuguhi standar penampilan orang lain di media sosial. Sejak kapan kamu mulai sering ngerasa insecure kayak gini?<|im_end|>
<|im_start|>user
Belakangan ini sih, semenjak sering scrolling reels sebelum tidur. Pas ngaca jadi ngerasa kurang in a way.<|im_end|>
<|im_start|>assistant
Pantas saja perasaanmu jadi nggak tenang akhir-akhir ini. Bagian mana dari diri kamu yang lagi paling bikin kamu nggak nyaman?<|im_end|>



In [3]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

# 1. Konfigurasi Kuantisasi 4-bit (QLoRA)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# 2. Muat Model Dasar ke Dual GPU T4 (Membuat variabel 'model')
print("Sedang memuat model Qwen ke memori GPU...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto"
)

# 3. Aktifkan Gradient Checkpointing
model.gradient_checkpointing_enable()
print("Model berhasil dimuat! Variabel 'model' kini sudah aktif.")


Sedang memuat model Qwen ke memori GPU...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Model berhasil dimuat! Variabel 'model' kini sudah aktif.


In [4]:
from peft import LoraConfig
from trl import SFTConfig

# 1. Konfigurasi LoRA Adapter untuk pengubahan gaya komunikasi
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# 2. Definisikan Argumen Menggunakan SFTConfig (Bawaan TRL Baru)
# Konfigurasi ini otomatis mengatur pembatasan loss hanya pada pesan asisten
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE_PER_DEVICE,
    gradient_accumulation_steps=GRADIENT_ACC_STEPS,
    learning_rate=LEARNING_RATE,
    num_train_epochs=EPOCHS,
    logging_steps=1,
    optim="paged_adamw_8bit",
    bf16=True,
    fp16=False,
    gradient_checkpointing=True,
    report_to="none",
    
    # KUNCI UTAMA: Menggantikan fungsi DataCollator lama secara native
    max_length=MAX_SEQ_LENGTH,       # Menggantikan max_seq_length
    dataset_text_field="text",       # Memindahkan pemetaan kolom ke sini
    packing=False,
    loss_type="nll"                  # Menghindari bug multi-GPU indeks tensor TRL
)

print("Konfigurasi LoRA dan SFTConfig berhasil didefinisikan secara native!")


Konfigurasi LoRA dan SFTConfig berhasil didefinisikan secara native!


In [7]:
import inspect
import types
from trl import SFTTrainer

# =====================================================================
# MONKEY PATCH: Mengatasi Bug TRL v1.13 + Qwen (functools.partial error)
# =====================================================================
class BoundMethodWrapper:
    def __init__(self, target_callable):
        self.target_callable = target_callable
        
    def __call__(self, *args, **kwargs):
        # Jalankan fungsi asli Qwen
        return self.target_callable(*args, **kwargs)
        
    @property
    def __func__(self):
        # Suntikkan properti __func__ yang dicari TRL dengan mengambil fungsi dasar di dalam partial
        return getattr(self.target_callable, "func", self.target_callable)

# Terapkan pembungkus ke fungsi forward model Qwen Anda
model.forward = BoundMethodWrapper(model.forward)
# =====================================================================

# 1. Inisialisasi Trainer (Sekarang akan lolos dari crash pengecekan TRL)
trainer = SFTTrainer(
    model=model,
    train_dataset=processed_dataset, 
    peft_config=peft_config,
    args=training_args
)

# 2. Eksekusi Pelatihan
print("Memulai proses fine-tuning Tone Alignment...")
trainer.train()

# 3. Amankan Hasil Adaptor LoRA ke folder kerja Kaggle
print("\nTraining selesai! Sedang menyimpan adaptor LoRA...")
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Adaptor berhasil disimpan di: {OUTPUT_DIR}")


Adding EOS to train dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Memulai proses fine-tuning Tone Alignment...


Step,Training Loss
1,3.703421
2,3.595306
3,3.580610
4,3.501082
5,3.530758
6,3.599434
7,3.404665
8,3.083977
9,3.363231
10,3.047606



Training selesai! Sedang menyimpan adaptor LoRA...
Adaptor berhasil disimpan di: /kaggle/working/qlora_qwen_empathy
